# Lab 08 — Segmentation: U-Net and SAM

**Deep Learning for Computer Vision · AIT**

Detection gives you a box. Segmentation gives you the **pixels** — and for a huge number
of real applications (medical imaging, crop analysis, satellite imagery, manufacturing
defect inspection) the box is not enough. If you want to measure *area*, you need a mask.

| Task | Question | Output |
|---|---|---|
| **Semantic** segmentation | which class is each pixel? | one label map; two adjacent cats merge |
| **Instance** segmentation | which *object* is each pixel? | one mask per object |
| **Panoptic** segmentation | both, jointly | instance masks for things, label map for stuff |

We build **U-Net** from scratch — still the default architecture for dense prediction
after ten years — and then use **SAM**, a promptable foundation model that segments
objects it was never trained on.

> **Build this carefully.** The U-Net you write here is reused almost unchanged as the
> denoising backbone in Lab 09. Encoder–decoder with skip connections is the same shape
> whether you are predicting a mask or predicting noise.

**Runs on:** Colab free T4, ~20 min.

**Reading:** [U-Net](https://arxiv.org/abs/1505.04597) ·
[Fully Convolutional Networks](https://arxiv.org/abs/1411.4038) ·
[SAM](https://arxiv.org/abs/2304.02643) ·
[SAM 2](https://arxiv.org/abs/2408.00714) ·
[Mask2Former](https://arxiv.org/abs/2112.01527)

## 0. Setup

In [ ]:
!pip install -q transformers timm

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt, time
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16          # T4 = Turing: fp16, not bf16
torch.manual_seed(0); np.random.seed(0)
print("device:", device)

### Dataset: Oxford-IIIT Pet trimaps

Every image comes with a **trimap**: pixel value 1 = the animal, 2 = background,
3 = boundary/ambiguous. We collapse it to a binary foreground mask. This is a genuine
segmentation dataset at a size a T4 can handle.

If the Oxford mirror is unavailable, the cell generates a synthetic shapes dataset so the
lab still runs end to end.

In [ ]:
IMG_SIZE = 128


class PetSeg(torch.utils.data.Dataset):
    def __init__(self, split="trainval", size=IMG_SIZE, limit=None):
        self.ds = torchvision.datasets.OxfordIIITPet(
            "./data", split=split, target_types="segmentation", download=True)
        self.size = size
        self.n = min(limit, len(self.ds)) if limit else len(self.ds)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        img, trimap = self.ds[i]
        img = img.convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        trimap = trimap.resize((self.size, self.size), Image.NEAREST)
        x = T.functional.to_tensor(img)
        x = T.functional.normalize(x, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        t = torch.from_numpy(np.array(trimap)).long()
        y = (t == 1).float().unsqueeze(0)        # 1 = pet, everything else = background
        return x, y


class SyntheticShapes(torch.utils.data.Dataset):
    # Fallback: random ellipses on noisy backgrounds. Same interface.
    def __init__(self, n=800, size=IMG_SIZE):
        self.n, self.size = n, size

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        g = np.random.default_rng(i)
        s = self.size
        yy, xx = np.mgrid[0:s, 0:s]
        cx, cy = g.integers(s // 4, 3 * s // 4, 2)
        rx, ry = g.integers(s // 8, s // 3, 2)
        mask = (((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2) < 1
        img = g.normal(0.5, 0.15, (3, s, s))
        img[:, mask] += g.uniform(0.4, 0.8)
        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask[None], dtype=torch.float32)


try:
    train_set = PetSeg("trainval", limit=2000)
    val_set = PetSeg("test", limit=400)
    DATASET = "OxfordIIITPet"
except Exception as e:
    print("Pet dataset unavailable (%s); using synthetic shapes." % type(e).__name__)
    train_set, val_set, DATASET = SyntheticShapes(800), SyntheticShapes(200), "SyntheticShapes"

# Worker processes must be forked: Python 3.14's default "forkserver" (and "spawn" on
# Windows/macOS) start a fresh interpreter that cannot see classes defined in this notebook.
import multiprocessing as mp
FORK = "fork" in mp.get_all_start_methods()
loader_kw = dict(batch_size=16, num_workers=2 if FORK else 0,
                 multiprocessing_context="fork" if FORK else None)
train_loader = torch.utils.data.DataLoader(train_set, shuffle=True, **loader_kw)
val_loader   = torch.utils.data.DataLoader(val_set,   shuffle=False, **loader_kw)
print("%s | %d train, %d val" % (DATASET, len(train_set), len(val_set)))

In [ ]:
def denorm(x):
    if DATASET == "OxfordIIITPet":
        m, s = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
        return np.clip(x.numpy().transpose(1, 2, 0) * s + m, 0, 1)
    return np.clip(x.numpy().transpose(1, 2, 0), 0, 1)


xb, yb = next(iter(train_loader))
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for j in range(6):
    ax[0, j].imshow(denorm(xb[j])); ax[0, j].axis("off")
    ax[1, j].imshow(yb[j, 0], cmap="gray"); ax[1, j].axis("off")
ax[0, 0].set_ylabel("image"); ax[1, 0].set_ylabel("mask")
plt.suptitle("%s - images and ground-truth masks" % DATASET); plt.tight_layout(); plt.show()

---
# 1. Why classification architectures fail here

A classifier ends with global average pooling and a linear layer: it deliberately throws
away *where*. For segmentation you need an output the same size as the input, with one
prediction per pixel.

The naive fix — remove the pooling and upsample the final 4x4 feature map back to
128x128 — produces blobs. By the time you reach the deepest layer you know *what* is in
the image but have lost all fine spatial detail.

**U-Net's answer:** an encoder that downsamples (gaining semantics, losing detail), a
decoder that upsamples, and **skip connections** that copy the high-resolution encoder
features across to the matching decoder stage.

```
input 128 ──[enc1]──┬─────────────────────────────► [dec1] ── output 128
                    │                                  ▲
                 64 ─[enc2]──┬──────────────► [dec2] ──┘
                             │                  ▲
                          32 ─[enc3]──┬─► [dec3]┘
                                      │     ▲
                                   16 ─[bottleneck]
```

The decoder gets semantics from below and detail from the side. This is the same
"give the gradient and the information a shortcut" idea as ResNet, applied across
resolutions instead of across depth.

---
# 2. U-Net from scratch

In [ ]:
class DoubleConv(nn.Module):
    # (conv -> norm -> activation) x 2. The basic unit of every U-Net stage.
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(cin, cout))

    def forward(self, x):
        return self.pool_conv(x)


class Up(nn.Module):
    def __init__(self, cin, skip_ch, cout):
        super().__init__()
        self.up = nn.ConvTranspose2d(cin, cin // 2, 2, stride=2)
        self.conv = DoubleConv(cin // 2 + skip_ch, cout)

    def forward(self, x, skip):
        x = self.up(x)
        # Pad if odd input sizes made the shapes disagree by one pixel.
        dy, dx = skip.size(2) - x.size(2), skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([skip, x], dim=1))     # <-- the skip connection

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        self.inc   = DoubleConv(in_ch, b)          # 128 -> 128,  b
        self.down1 = Down(b, b * 2)                # 128 ->  64,  2b
        self.down2 = Down(b * 2, b * 4)            #  64 ->  32,  4b
        self.down3 = Down(b * 4, b * 8)            #  32 ->  16,  8b
        self.bott  = Down(b * 8, b * 8)            #  16 ->   8,  8b  (bottleneck)

        self.up1 = Up(b * 8, b * 8, b * 4)
        self.up2 = Up(b * 4, b * 4, b * 2)
        self.up3 = Up(b * 2, b * 2, b)
        self.up4 = Up(b, b, b)
        self.outc = nn.Conv2d(b, out_ch, 1)        # 1x1 conv = per-pixel classifier

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.bott(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)                        # logits, same H x W as input


net = UNet()
out = net(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print("input :", (2, 3, IMG_SIZE, IMG_SIZE))
print("output:", tuple(out.shape))
print("params (M): %.2f" % (sum(p.numel() for p in net.parameters()) / 1e6))

---
# 3. Losses and metrics for dense prediction

Pixel-wise cross-entropy alone is a poor choice when the foreground is small. If 95% of
pixels are background, predicting "background everywhere" scores 95% accuracy and zero
usefulness.

**Dice loss** measures *overlap* instead, and is insensitive to that imbalance:

$$\text{Dice} = \frac{2|P \cap G|}{|P| + |G|}$$

In practice: use **BCE + Dice**. BCE gives well-behaved per-pixel gradients everywhere;
Dice supplies the region-level objective you actually care about.

For reporting, **IoU** (a.k.a. Jaccard) is the standard. Note Dice and IoU always agree
in ordering — Dice = 2·IoU / (1 + IoU) — but Dice reads higher, so always say which one
you are quoting.

In [ ]:
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    num = 2 * (p * target).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return (1 - num / den).mean()


def bce_dice(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)


@torch.no_grad()
def iou_score(logits, target, thr=0.5):
    p = (torch.sigmoid(logits) > thr).float()
    inter = (p * target).sum(dim=(1, 2, 3))
    union = ((p + target) > 0).float().sum(dim=(1, 2, 3))
    return (inter / (union + 1e-7)).mean().item()


# Sanity check: a perfect prediction should give ~0 loss and IoU 1.
t = (torch.rand(2, 1, 32, 32) > 0.7).float()
perfect = (t * 20 - 10)
print("perfect  -> dice loss %.4f | IoU %.4f" % (dice_loss(perfect, t).item(), iou_score(perfect, t)))
print("inverted -> dice loss %.4f | IoU %.4f" % (dice_loss(-perfect, t).item(), iou_score(-perfect, t)))

In [ ]:
def train_unet(model, epochs=8, lr=3e-4):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader), pct_start=0.25)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    hist = {"loss": [], "iou": []}
    t0 = time.time()

    for ep in range(epochs):
        model.train(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
                loss = bce_dice(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * x.size(0)

        model.eval(); ious = []
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
            for x, y in val_loader:
                ious.append(iou_score(model(x.to(device)).float(), y.to(device)))
        hist["loss"].append(running / len(train_set)); hist["iou"].append(float(np.mean(ious)))
        print("  epoch %d/%d  loss %.4f  val IoU %.4f" % (ep + 1, epochs, hist["loss"][-1], hist["iou"][-1]))

    print("  %.0fs | best IoU %.4f" % (time.time() - t0, max(hist["iou"])))
    return hist


unet = UNet(base=32)
hist = train_unet(unet, epochs=8)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["loss"], marker="o"); ax[0].set_title("train loss (BCE + Dice)")
ax[1].plot(hist["iou"], marker="o", color="tab:green"); ax[1].set_title("val IoU")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
unet.eval()
xb, yb = next(iter(val_loader))
with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
    pred = torch.sigmoid(unet(xb.to(device)).float()).cpu()

fig, ax = plt.subplots(3, 6, figsize=(15, 7.5))
for j in range(6):
    ax[0, j].imshow(denorm(xb[j]));            ax[0, j].axis("off")
    ax[1, j].imshow(yb[j, 0], cmap="gray");    ax[1, j].axis("off")
    ax[2, j].imshow(pred[j, 0] > 0.5, cmap="gray"); ax[2, j].axis("off")
    ax[2, j].set_title("IoU %.2f" % iou_score(pred[j:j+1].logit(eps=1e-6), yb[j:j+1]), fontsize=9)
for i, name in enumerate(["image", "ground truth", "U-Net"]):
    ax[i, 0].set_ylabel(name)
plt.suptitle("U-Net predictions"); plt.tight_layout(); plt.show()

---
# 4. SAM: segmentation without training

[Segment Anything](https://arxiv.org/abs/2304.02643) is a **promptable** segmentation
foundation model. You give it an image plus a prompt — a point, a box, a rough mask — and
it returns a mask for whatever object that prompt indicates. It was trained on 1.1 billion
masks, and it generalises to categories it has never seen, because it was never asked to
name anything. It only ever learned "what is an object here".

The architecture is a deliberate speed trade-off:

- a heavy **ViT image encoder** runs **once** per image (expensive),
- a tiny **prompt encoder** + **mask decoder** run per prompt (milliseconds).

So interactive labelling is fast even though the backbone is large.

**The lineage:** SAM (2023) does images; **SAM 2** (2024) adds video with a memory
module so masks track across frames; Meta's later releases extend prompting to text
concepts. We use SAM 1 here because it installs cleanly from `transformers` with no extra
dependencies — the prompting interface is the same across versions.

In [ ]:
from transformers import SamModel, SamProcessor
import requests
from io import BytesIO

sam = SamModel.from_pretrained("facebook/sam-vit-base").to(device).eval()
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
print("SAM params (M): %.0f" % (sum(p.numel() for p in sam.parameters()) / 1e6))

url = "https://huggingface.co/ybelkada/segment-anything/resolve/main/assets/car.png"
raw_image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
plt.figure(figsize=(8, 5)); plt.imshow(raw_image); plt.axis("off")
plt.title("input - %dx%d" % raw_image.size); plt.show()

In [ ]:
def sam_predict(image, input_points=None, input_boxes=None):
    inputs = processor(image, input_points=input_points, input_boxes=input_boxes,
                       return_tensors="pt").to(device)
    with torch.no_grad():
        out = sam(**inputs, multimask_output=True)
    masks = processor.image_processor.post_process_masks(
        out.pred_masks.cpu(), inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu())
    return masks[0][0].numpy(), out.iou_scores[0][0].cpu().numpy()


def show_masks(image, masks, scores, title=""):
    fig, ax = plt.subplots(1, len(masks) + 1, figsize=(4.5 * (len(masks) + 1), 4.5))
    ax[0].imshow(image); ax[0].set_title("input"); ax[0].axis("off")
    for i, (m, s) in enumerate(zip(masks, scores)):
        ax[i + 1].imshow(image)
        overlay = np.zeros((*m.shape, 4)); overlay[m] = [0.1, 0.6, 1.0, 0.6]
        ax[i + 1].imshow(overlay)
        ax[i + 1].set_title("mask %d - predicted IoU %.3f" % (i, s)); ax[i + 1].axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
# --- Prompt 1: a single point on the car ---
point = [[[450, 600]]]                       # [batch][per-image][x, y]
masks, scores = sam_predict(raw_image, input_points=point)

plt.figure(figsize=(6, 4)); plt.imshow(raw_image)
plt.scatter([450], [600], c="lime", s=200, marker="*", edgecolors="black")
plt.axis("off"); plt.title("the prompt"); plt.show()

show_masks(raw_image, masks, scores, "Point prompt -> SAM returns 3 valid interpretations")

Notice SAM returns **three** masks for one point, with predicted IoU scores. A point is
genuinely ambiguous — clicking a car wheel could mean the tyre, the wheel assembly, or the
whole car. Rather than guessing, SAM exposes the ambiguity at three scales and lets you
pick. This is a design lesson worth remembering: when the input underdetermines the
output, return the distribution, not a point estimate.

A **box** prompt is far less ambiguous, which is why box-prompted SAM is the workhorse of
auto-labelling pipelines.

In [ ]:
# --- Prompt 2: a box around the car ---
box = [[[75, 275, 1725, 850]]]               # [batch][per-image][x1, y1, x2, y2]
masks_b, scores_b = sam_predict(raw_image, input_boxes=box)
show_masks(raw_image, masks_b[:1], scores_b[:1], "Box prompt -> unambiguous")

### The pattern that matters: detector → SAM → your model

You rarely deploy SAM directly (it is large, and it does not know class names). The
high-value pattern is **auto-labelling**:

1. Run a detector (Lab 07) to get boxes and class labels.
2. Feed each box to SAM as a prompt to get a pixel-accurate mask, for free.
3. Train your own small, fast U-Net on those masks.

You end up with a specialised model that runs 100x faster than SAM, trained on labels you
never paid a human to draw. Exercise 5 walks you through it.

In [ ]:
# Detector -> SAM, wired together end to end.
from ultralytics import YOLO

det = YOLO("yolo11n.pt")
res = det.predict(raw_image, conf=0.3, verbose=False)[0]
boxes_xyxy = res.boxes.xyxy.cpu().numpy().tolist()
labels = [res.names[int(c)] for c in res.boxes.cls]
print("detector found:", labels)

if boxes_xyxy:
    masks_auto, _ = sam_predict(raw_image, input_boxes=[boxes_xyxy])
    plt.figure(figsize=(9, 5)); plt.imshow(raw_image)
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(masks_auto), 1)))
    for m, c, lab in zip(masks_auto, colors, labels):
        mm = m if m.ndim == 2 else m[0]
        ov = np.zeros((*mm.shape, 4)); ov[mm.astype(bool)] = [*c[:3], 0.55]
        plt.imshow(ov)
    plt.axis("off"); plt.title("YOLO boxes -> SAM masks: automatic instance segmentation")
    plt.show()

---
# Exercises

**1. Skip connections are the whole point.** Delete the `torch.cat([skip, x])` from `Up`
(concatenate nothing; just upsample) and retrain. Report the IoU drop and show a
side-by-side prediction. Describe *what kind* of error appears — is it the object's
location or its boundary that degrades?

**2. Loss ablation.** Train three U-Nets with (a) BCE only, (b) Dice only, (c) BCE+Dice.
Compare IoU and, more importantly, look at the predicted masks. Which loss produces
over-confident blobs, and which produces ragged edges? Then construct a dataset where the
foreground is under 2% of pixels and repeat — does your ranking change?

**3. Capacity and receptive field.** Train U-Nets with `base` = 16, 32 and 64. Plot IoU
against parameter count and training time. Where does it saturate on this dataset?

**4. Multi-class segmentation.** Extend `UNet` to `out_ch=3` and predict the full Pet
trimap (pet / background / boundary) with cross-entropy. Report per-class IoU. Why is the
boundary class so much harder, and is that a model failure or a label-quality issue?

**5. Auto-labelling pipeline.** Take 100 unlabelled images of one object class from your
own Lab 04/07 dataset. Use YOLO + SAM to generate masks automatically, then train your
U-Net on them. Compare against 20 masks you draw by hand. Which gives the better model,
and how many hand labels would it take to beat the automatic ones?

**6. Boundary-aware evaluation.** IoU is dominated by the interior of large objects and
barely notices boundary errors. Implement **Boundary IoU** (IoU computed only within a
d-pixel band of the mask contour) and re-rank your models from Exercise 2. Does the
ranking change? Which metric would you report to a customer measuring leaf area?

**7. SAM's ambiguity.** Prompt SAM with points at three different depths of structure in
one image (a wheel, a window, a whole vehicle). For each, show all three returned masks
and their predicted IoU. When does SAM's own confidence correctly identify the mask you
wanted, and when does it not?